# Week 9: Ground-to-satellite line-of-sight and coverage

**Track:** Orbital Analyst (Intermediate)  
**Full primer + quiz:** [https://launchdetect.com/academy/week/9/](https://launchdetect.com/academy/week/9/)  
**Track index:** [https://launchdetect.com/academy/orbital-analyst/](https://launchdetect.com/academy/orbital-analyst/)

---

_Can a ground station see a given satellite right now? This is a question of geometry, line-of-sight, and (a little) atmosphere. This week you build the math and the code._


## Why this week matters

Knowing where a satellite IS in 3D space (Week 8) is half the story. The other half is **whether anyone on the ground can see it**. That's a geometric question about line-of-sight from a moving point to a fixed point — and it's the question that drives every satellite-spotting app, every ground-station planner, every communications-window scheduler.

Practically, the question shows up two ways: *given a satellite, when is it visible from my location?* (consumer-app use case) and *given a ground station, when can it see the satellite?* (operations use case). Same math, different starting point. This week we do both.

**Pass prediction is also the foundation for Week 18's Cesium 3D globe.** A 3D visualization is only as good as the underlying ephemeris + visibility logic. Build the math now, animate it later.


## Learning objectives

By the end of this lab you will be able to:

- Compute look angles (azimuth, elevation) from a ground station to a satellite
- Identify pass start, max-elevation, and pass end times
- Build a coverage cone polygon for a given sensor swath
- Account for atmospheric refraction at low elevation angles


## Setup — and why these dependencies

Continues with `skyfield`. No new deps — pass prediction is built in.


In [ ]:
# Install required packages
!pip install -q leafmap[common] skyfield geopandas shapely


## The approach (and why)

Use `skyfield.EarthSatellite.find_events(observer, t0, t1, altitude_degrees=10.0)` to enumerate every visibility event (rise, culminate, set) in a 24-hour window. For each event, compute the look angles (azimuth, elevation).

**Why 10° elevation threshold?** Below 10° atmospheric refraction and terrain obstruction (buildings, trees) make detection unreliable. Most practical apps use 10° as the minimum 'visible' threshold. For naked-eye satellite spotting in dark sky, 30° is more useful.

**Why rise / culminate / set as the event triple?** Together they fully describe a pass: when it starts, when it's highest, when it ends. Most consumer apps only show the culmination time and elevation (the 'peak visibility' moment), but the full triple matters for ground-station ops.


In [ ]:
# Week 9: next ISS pass over a user-supplied location.
from skyfield.api import EarthSatellite, load, wgs84

tle1 = "1 25544U 98067A   24130.50145833  .00018539  00000-0  33188-3 0  9994"
tle2 = "2 25544  51.6406 348.5395 0006703 117.9568 358.1729 15.50289267449420"
ts = load.timescale()
iss = EarthSatellite(tle1, tle2, "ISS", ts)

# Observer: New York City (replace with your lat/lon)
observer = wgs84.latlon(40.7128, -74.0060)

t0 = ts.now()
t1 = ts.tt_jd(t0.tt + 1.0)  # next 24 hours
times, events = iss.find_events(observer, t0, t1, altitude_degrees=10.0)

EVENT_NAMES = {0: "rise (AOS)", 1: "culminate (TCA)", 2: "set (LOS)"}
for t, ev in zip(times, events):
    alt, az, _ = (iss - observer).at(t).altaz()
    print(f"{t.utc_iso()}  {EVENT_NAMES[ev]:<18}  alt={alt.degrees:6.2f}°  az={az.degrees:6.2f}°")

# TODO: filter for passes with culmination > 30° elevation. Output the next
# 3 visible passes as a JSON record.


## What just happened — and why it works

`find_events()` walks the propagator's output and detects elevation crossings: going from below 10° to above 10° (rise), reaching local max (culminate), going from above 10° to below 10° (set). The returned arrays are time-sorted.

For each event we compute the apparent position as seen from the observer: `(iss - observer).at(t).altaz()` gives altitude (elevation in skyfield's vocabulary) and azimuth (compass direction). Altitude 0° = horizon, 90° = directly overhead.

Notice the altaz call doesn't apply atmospheric refraction by default. For sub-0.5° accuracy near the horizon, pass `topos.altaz(refraction=True)`.


## Common gotchas

- **Elevation/altitude terminology collision.** Astronomers say 'altitude' for the vertical angle above horizon. Aviation says 'elevation'. Geodesy says 'altitude' for height above ellipsoid. Context tells you which is meant.
- **Azimuth from true north vs magnetic north.** skyfield returns true-north azimuth. A magnetic compass shows magnetic-north azimuth. They differ by the local declination (up to 25° in some places).
- **Terrain obstruction.** A satellite at 8° elevation due south might be blocked by a mountain. Production apps use a 'horizon mask' — a per-azimuth minimum elevation derived from local terrain.


## Self-check

Verify your solution against these checks before considering the lab complete:

- [ ] Output type matches the expected format (GeoJSON / PNG / table / etc.).
- [ ] No exceptions raised on a clean run.
- [ ] Result is visually plausible — open the map cell, scan the values, sanity-check magnitudes.
- [ ] Quiz on the [week page](https://launchdetect.com/academy/week/9/) — try answering before checking the key.

---

Found a bug or want to contribute an improvement? Open an issue or PR at [github.com/launchdetect/academy-labs](https://github.com/launchdetect/academy-labs).
